In [ ]:
# ============================================================
# STANDARD LIBRARY - Basis-Funktionen
# ============================================================
import asyncio           # Asynchrone Programmierung (async/await)
import configparser      # Konfigurationsdateien lesen (secret.cfg)
import hashlib           # Kryptographische Hash-Funktionen
import hmac              # Authentifizierung/Signatur für API-Requests
import json              # JSON Daten verarbeiten
import math              # Mathematische Funktionen
import queue             # Thread-sichere Warteschlangen
import threading         # Multithreading für parallele Verarbeitung
import time              # Zeit/Verzögerungen

from dataclasses import dataclass           # Datenklassen definieren
from datetime import datetime, timezone     # Datum und Zeit
from decimal import Decimal, ROUND_DOWN     # Dezimalzahlen mit hoher Präzision
from pathlib import Path                    # Cross-plattform Dateipfade
from typing import Iterable, List, Optional # Typ-Hinweise für besseren Code
from __future__ import annotations          # Future type annotations


# ============================================================
# EXTERNE BIBLIOTHEKEN - Datenverarbeitung & Analyse
# ============================================================
import numpy as np                           # Numerische Berechnungen
import pandas as pd                          # Datenanalyse und TimeSeriesManagement
import ccxt                                  # Kryptobörsen-API (Binance etc.)

from scipy.cluster.hierarchy import linkage, fcluster  # Hierarchisches Clustering
from scipy.spatial.distance import squareform          # Distanzmatrizen
from sklearn.preprocessing import StandardScaler        # Daten normalisieren
from ripser import ripser                   # Topologische Datenanalyse (TDA)


# ============================================================
# EXTERNE BIBLIOTHEKEN - Netzwerk & Web
# ============================================================
import requests          # HTTP Requests (REST API)
import websockets        # WebSocket Clients (Echtzeit-Datenströme)

from dateutil.relativedelta import relativedelta  # Relative Datums-Arithmetik


# ============================================================
# EXTERNE BIBLIOTHEKEN - Visualisierung
# ============================================================
import matplotlib
matplotlib.use("TkAgg")  # Tkinter Backend für Plotting
import matplotlib.pyplot as plt                         # Graphiken/Plots
from matplotlib.animation import FuncAnimation          # Animierte Plots

## Config ##


In [2]:
INTERVAL = "1m"
TICKERS = ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'SOLUSDT', 'XRPUSDT']  # expand to 40+
symbol = TICKERS[0] # e.g. 'BTCUSDT'

TESTNET_REST = "https://testnet.binance.vision"
streams = [f"{ticker.lower()}@kline_{INTERVAL}" for ticker in TICKERS]
stream_path = "/".join(streams)
ws_url = f"wss://stream.testnet.binance.vision/stream?streams={stream_path}"

PRELOAD_LIMIT = 200
WINDOW = 250

# Strategie: SMA Cross + RSI Filter
SMA_SHORT = 5
SMA_LONG = 20
RSI_PERIOD = 14
RSI_BUY_MAX = 75
RSI_SELL_MIN = 25

# Trading (Testnet)
TRADE_ENABLED = True
USDT_PER_BUY = 100.0
RECV_WINDOW = 10000  # vorher 5000, etwas höher = weniger -1021 Stress

# Plot-Update: jede Sekunde checken, gezeichnet wird nur wenn neue Kerze da ist
PLOT_UPDATE_MS = 1000

# Balances/Prints (real für Testnet Guthaben)
BALANCE_EVERY_N_CANDLES = 1
PRINT_EVERY_CLOSED_CANDLE = True

# Portfolio-Simulation (Performance-Visualisierung)
START_USDT_SIM = 1000.0
BUY_USDT_SIM = 100.0


# ============================================================
# BINANCE AUTH / REST + TIME OFFSET
# ============================================================
@dataclass
class BinanceKeys:
    api_key: str
    secret_key: str


# globaler Zeit-Offset (Serverzeit - lokale Zeit) in ms
TIME_OFFSET_MS = 0

In [ ]:
# =========================
# Konfiguration
# =========================
BASE_URL = "https://data-api.binance.vision"
INTERVAL = "1h"          # z.B. "1m", "5m", "1h", "1d"
MONTHS_BACK = 12
MIN_QUOTE_VOLUME = 50_000_000  # 50 Mio USD Gegenwert in 24h
QUOTE_ASSET = "USDT"
OUTPUT_DIR = Path("data/klines")
REQUEST_TIMEOUT = 20
SLEEP_BETWEEN_REQUESTS = 0.15

session = requests.Session()


# =========================
# Hilfsfunktionen
# =========================
def binance_get(path: str, params: Optional[dict] = None):
    url = f"{BASE_URL}{path}"
    r = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    return r.json()


def get_exchange_info() -> dict:
    return binance_get("/api/v3/exchangeInfo")


def get_all_ticker_24h() -> list[dict]:
    return binance_get("/api/v3/ticker/24hr")


def get_klines(symbol: str, interval: str, start_ms: int, end_ms: int, limit: int = 1000) -> list[list]:
    params = {
        "symbol": symbol,
        "interval": interval,
        "startTime": start_ms,
        "endTime": end_ms,
        "limit": limit,
    }
    return binance_get("/api/v3/klines", params=params)


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def dt_to_ms(dt: datetime) -> int:
    return int(dt.timestamp() * 1000)


def interval_to_milliseconds(interval: str) -> int:
    unit = interval[-1]
    value = int(interval[:-1])

    multipliers = {
        "s": 1000,
        "m": 60 * 1000,
        "h": 60 * 60 * 1000,
        "d": 24 * 60 * 60 * 1000,
        "w": 7 * 24 * 60 * 60 * 1000,
    }

    if unit == "M":
        # Monatskerzen sind unregelmässig; für Pagination nicht ideal.
        raise ValueError("Intervall 'M' wird für diese Pagination nicht unterstützt.")

    if unit not in multipliers:
        raise ValueError(f"Unbekanntes Intervall: {interval}")

    return value * multipliers[unit]


# =========================
# 1) Symboluniversum vorfiltern
# =========================
def filter_symbols_binance_only(
    exchange_info: dict,
    ticker_24h: list[dict],
    quote_asset: str = "USDT",
    min_quote_volume: float = 50_000_000,
) -> list[str]:
    volume_map = {}
    for row in ticker_24h:
        symbol = row.get("symbol")
        quote_volume = float(row.get("quoteVolume", 0.0))
        volume_map[symbol] = quote_volume

    selected = []

    for sym in exchange_info["symbols"]:
        symbol = sym["symbol"]
        status = sym.get("status")
        quote = sym.get("quoteAsset")
        permissions = sym.get("permissions", [])

        if status != "TRADING":
            continue
        if quote != quote_asset:
            continue
        if "SPOT" not in permissions:
            continue
        if symbol not in volume_map:
            continue
        if volume_map[symbol] < min_quote_volume:
            continue

        selected.append(symbol)

    return sorted(selected)


# =========================
# 2) Historische Klines seitenweise laden
# =========================
def fetch_12_months_klines(symbol: str, interval: str, months_back: int = 12) -> pd.DataFrame:
    end_dt = utc_now()
    start_dt = end_dt - relativedelta(months=months_back)

    start_ms = dt_to_ms(start_dt)
    end_ms = dt_to_ms(end_dt)

    step_ms = interval_to_milliseconds(interval)
    all_rows: list[list] = []

    current_start = start_ms

    while current_start < end_ms:
        batch = get_klines(
            symbol=symbol,
            interval=interval,
            start_ms=current_start,
            end_ms=end_ms,
            limit=1000
        )

        if not batch:
            break

        all_rows.extend(batch)

        last_open_time = batch[-1][0]
        next_start = last_open_time + step_ms

        if next_start <= current_start:
            break

        current_start = next_start
        time.sleep(SLEEP_BETWEEN_REQUESTS)

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(
        all_rows,
        columns=[
            "open_time", "open", "high", "low", "close", "volume",
            "close_time", "quote_asset_volume", "number_of_trades",
            "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
        ]
    )

    # Typen bereinigen
    numeric_cols = [
        "open", "high", "low", "close", "volume",
        "quote_asset_volume", "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume"
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    int_cols = ["open_time", "close_time", "number_of_trades"]
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    df["symbol"] = symbol
    df["interval"] = interval
    df["open_datetime"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["year"] = df["open_datetime"].dt.year.astype("int16")
    df["month"] = df["open_datetime"].dt.month.astype("int8")

    # Duplikate vermeiden
    df = df.drop_duplicates(subset=["symbol", "interval", "open_time"]).sort_values("open_time").reset_index(drop=True)

    # Spalte "ignore" löschen
    df = df.drop(columns=["ignore"])

    return df


# =========================
# 3) Parquet speichern
# =========================
def save_partitioned_parquet(df: pd.DataFrame, root_dir: Path) -> None:
    if df.empty:
        return

    root_dir.mkdir(parents=True, exist_ok=True)

    # partition_cols erzeugen Verzeichnisstruktur:
    # interval=.../symbol=.../year=.../month=...
    df.to_parquet(
        root_dir,
        engine="pyarrow",
        compression="zstd",
        index=False,
        partition_cols=["interval", "symbol", "year", "month"]
    )


# =========================
# 4) Optional: Symbolliste speichern
# =========================
def save_selected_symbols(symbols: list[str], output_file: Path) -> None:
    output_file.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"symbol": symbols}).to_csv(output_file, index=False)


# =========================
# 5) Hauptlauf
# =========================
def main():
    print("Lade Exchange-Informationen...")
    exchange_info = get_exchange_info()

    print("Lade 24h-Ticker-Statistiken...")
    ticker_24h = get_all_ticker_24h()

    print("Filtere Symbole...")
    selected_symbols = filter_symbols_binance_only(
        exchange_info=exchange_info,
        ticker_24h=ticker_24h,
        quote_asset=QUOTE_ASSET,
        min_quote_volume=MIN_QUOTE_VOLUME
    )

    print(f"Anzahl gefilterter Symbole: {len(selected_symbols)}")
    print(selected_symbols[:20])

    save_selected_symbols(selected_symbols, Path("data/selected_symbols.csv"))

    all_frames = []

    for i, symbol in enumerate(selected_symbols, start=1):
        print(f"[{i}/{len(selected_symbols)}] Lade {symbol} ...")
        try:
            df = fetch_12_months_klines(symbol=symbol, interval=INTERVAL, months_back=MONTHS_BACK)
            if not df.empty:
                all_frames.append(df)
        except Exception as e:
            print(f"Fehler bei {symbol}: {e}")

    if not all_frames:
        print("Keine Daten geladen.")
        return

    full_df = pd.concat(all_frames, ignore_index=True)
    save_partitioned_parquet(full_df, OUTPUT_DIR)

    print("Fertig.")
    print(f"Gespeichert unter: {OUTPUT_DIR.resolve()}")
    print(f"Zeilen gesamt: {len(full_df):,}")

In [ ]:
def load_keys(cfg_path: str = "secret.cfg") -> BinanceKeys:
    cfg = configparser.ConfigParser()
    if not cfg.read(cfg_path):
        raise FileNotFoundError(f"Konnte {cfg_path} nicht lesen.")
    return BinanceKeys(
        api_key=cfg["BINANCE"]["TEST_API_KEY"].strip(),
        secret_key=cfg["BINANCE"]["TEST_SECRET_KEY"].strip()
    )


def _local_ms() -> int:
    return int(time.time() * 1000)


def _ts_ms() -> int:
    # Server synchronisiert: lokale Zeit + Offset
    return _local_ms() + TIME_OFFSET_MS


def _sign(secret_key: str, query_string: str) -> str:
    return hmac.new(
        secret_key.encode("utf-8"),
        query_string.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()


def public_get(path: str, params: dict | None = None):
    url = f"{TESTNET_REST}{path}"
    r = requests.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Public Fehler {r.status_code}: {r.text}")
    return r.json()


def sync_server_time_offset():
    """
    Holt die Binance-Serverzeit und berechnet den Offset.
    Offset = serverTime - localTime
    """
    global TIME_OFFSET_MS
    try:
        server = public_get("/api/v3/time")
        server_ms = int(server["serverTime"])
        local_ms = _local_ms()
        TIME_OFFSET_MS = server_ms - local_ms
        print(f"[TIME] Offset synchronisiert: {TIME_OFFSET_MS} ms (server-local)")
    except Exception as e:
        print(f"[TIME] Konnte Offset nicht synchronisieren: {e}")


def signed_request(keys: BinanceKeys, method: str, path: str, params: dict):
    """
    Signierte REST Requests mit Zeitoffset-Fix.
    Falls -1021 kommt, wird Offset neu synchronisiert und 1x retry gemacht.
    """
    params = dict(params or {})
    params["timestamp"] = _ts_ms()
    params["recvWindow"] = RECV_WINDOW

    query = "&".join([f"{k}={params[k]}" for k in sorted(params.keys())])
    signature = _sign(keys.secret_key, query)

    url = f"{TESTNET_REST}{path}?{query}&signature={signature}"
    headers = {"X-MBX-APIKEY": keys.api_key}

    r = requests.request(method=method, url=url, headers=headers, timeout=20)

    if r.status_code >= 400:
        # Versuch: wenn Timestamp-Problem, Offset neu sync + retry
        try:
            err = r.json()
        except Exception:
            err = {"raw": r.text}

        if isinstance(err, dict) and err.get("code") == -1021:
            print("[TIME] -1021 Timestamp-Fehler -> resync und retry")
            sync_server_time_offset()

            params["timestamp"] = _ts_ms()
            query = "&".join([f"{k}={params[k]}" for k in sorted(params.keys())])
            signature = _sign(keys.secret_key, query)
            url = f"{TESTNET_REST}{path}?{query}&signature={signature}"

            r2 = requests.request(method=method, url=url, headers=headers, timeout=20)
            if r2.status_code >= 400:
                raise RuntimeError(f"Signed Fehler {r2.status_code}: {r2.text}")
            return r2.json()

        raise RuntimeError(f"Signed Fehler {r.status_code}: {r.text}")

    return r.json()

In [ ]:
# 1. Fetch data
# ======================
exchange = ccxt.binance()
tickers = ['BTC/USDT', 'ETH/USDT', 'BNB/USDT', 'SOL/USDT', 'XRP/USDT']  # expand to 40+

def fetch_returns(tickers, days=500):
    dfs = []
    for symbol in tickers:
        ohlcv = exchange.fetch_ohlcv(symbol, '1d', limit=days)
        df = pd.DataFrame(ohlcv, columns=['timestamp','open','high','low','close','volume'])
        df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
        df.set_index('date', inplace=True)
        df['returns'] = df['close'].pct_change()
        dfs.append(df['returns'])
    data = pd.concat(dfs, axis=1)
    data.columns = tickers
    data.dropna(inplace=True)
    return data

returns = fetch_returns(tickers, days=500)
